<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---- Google Drive 연결 ----
from google.colab import drive
drive.mount('/content/drive')

# 파일 경로 확인 (업로드한 위치에 맞게)
# 보통 '내 드라이브' 최상단에 올렸으면 아래 경로야
import os
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/paderborn_features_scaled_no_overlap.csv'
print("파일 존재 여부:", os.path.exists(CSV_PATH))

import pandas as pd
df = pd.read_csv(CSV_PATH, nrows=5)   # 처음 5줄만 빠르게 미리보기
print("전체 컬럼:", list(df.columns))
print("\n행 개수 확인용 (전체):")
df_full = pd.read_csv(CSV_PATH, usecols=[0])  # 첫 컬럼만 읽어 행수 세기
print("전체 행 수:", len(df_full))

import pandas as pd

# CSV_PATH = '/content/drive/MyDrive/paderborn_features_scaled_no_overlap.csv' # <-- 이 줄이 잘못된 경로를 다시 설정하고 있었습니다.
# 위의 첫 번째 CSV_PATH 정의가 올바르므로, 이 줄은 제거하거나 주석 처리합니다.
# 또는 올바른 경로로 변경합니다.
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/paderborn_features_scaled_no_overlap.csv'

# 전체 컬럼을 한 줄씩 세로로 출력 (잘리지 않게)
df_head = pd.read_csv(CSV_PATH, nrows=5)
print("=" * 40)
print(f"총 컬럼 개수: {len(df_head.columns)}개")
print("=" * 40)
for i, col in enumerate(df_head.columns):
    print(f"{i:2d}. {col}")

# 전체 행 수 (크게 출력)
df_ids = pd.read_csv(CSV_PATH, usecols=[0])
print("\n" + "=" * 40)
print(f">>> 전체 행 수: {len(df_ids):,} 개 <<<")
print("=" * 40)

# 라벨로 보이는 컬럼이 있으면 그 값 분포도 확인
# (label, fault, class, type 중 하나가 들어간 컬럼 자동 탐색)
label_candidates = [c for c in df_head.columns
                    if any(k in c.lower() for k in ['label','fault','class','type','target','damage','condition'])]
print("\n[라벨로 추정되는 컬럼]:", label_candidates)

if label_candidates:
    lbl = label_candidates[0]
    df_lbl = pd.read_csv(CSV_PATH, usecols=[lbl])
    print(f"\n'{lbl}' 컬럼의 값 분포:")
    print(df_lbl[lbl].value_counts())

    import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/paderborn_features_scaled_no_overlap.csv')

# 1) 컬럼 전체 목록
print("=== COLUMNS ===")
for i, c in enumerate(df.columns):
    print(f"{i}: {c}")

# 2) 상위 5행 (전치해서 보기 편하게)
print("\n=== HEAD (transposed) ===")
print(df.head(5).T.to_string())

# 3) 수치 요약
print("\n=== DESCRIBE ===")
print(df.describe().T.to_string())

# 4) shape
print(f"\nSHAPE: {df.shape}")
# =========================================================
# AI 김반장 LITE - Phase A: Paderborn 이진분류 1D-CNN
# Healthy vs Fault | GroupKFold(누수 방지) | FNR/FAR 평가
# =========================================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (confusion_matrix, f1_score,
                             accuracy_score, classification_report)
import matplotlib.pyplot as plt

# ---------- 1. 데이터 로드 ----------
PATH = '/content/drive/MyDrive/Colab Notebooks/paderborn_features_scaled_no_overlap.csv'
df = pd.read_csv(PATH)
print("SHAPE:", df.shape)

# ---------- 2. 특징 / 라벨 / 그룹 정의 ----------
# 특징: vib_/cur1_/cur2_ 로 시작하는 컬럼만 (이미 스케일됨)
feat_cols = [c for c in df.columns
             if c.startswith(('vib_', 'cur1_', 'cur2_'))]
print(f"사용 특징 수: {len(feat_cols)}")   # 57개 예상

X = df[feat_cols].values.astype('float32')

# 라벨: Healthy=0, Real/Artificial=1 (이진)
y = (df['label'] != 'Healthy').astype('int32').values
print("라벨 분포(0=정상,1=결함):", np.bincount(y))

# 그룹: 데이터 누수 방지용 (같은 베어링은 한쪽에만)
groups = df['bearing_code'].values

# ---------- 3. 그룹 기반 + 클래스 층화 분할 ----------
# Paderborn: K0xx = 정상, KA/KI/KB = 결함
bearings = df.groupby('bearing_code')['label'].first()
healthy_b = [b for b, lab in bearings.items() if lab == 'Healthy']
fault_b   = [b for b, lab in bearings.items() if lab != 'Healthy']
print(f"정상 베어링 {len(healthy_b)}종: {healthy_b}")
print(f"결함 베어링 {len(fault_b)}종: {len(fault_b)}개")

import random
random.seed(42)
random.shuffle(healthy_b); random.shuffle(fault_b)

# 각 그룹의 20%를 테스트로 (정상이 적으니 최소 1종은 보장)
n_h_te = max(1, int(len(healthy_b) * 0.2))
n_f_te = max(1, int(len(fault_b)   * 0.2))
test_b  = set(healthy_b[:n_h_te] + fault_b[:n_f_te])
train_b = set(healthy_b[n_h_te:] + fault_b[n_f_te:])

print(f"\nTrain 정상: {[b for b in train_b if b in healthy_b]}")
print(f"Test  정상: {[b for b in test_b  if b in healthy_b]}")  # 비면 안 됨!

tr_idx = df['bearing_code'].isin(train_b).values
te_idx = df['bearing_code'].isin(test_b).values

X_tr, X_te = X[tr_idx], X[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]
print(f"\nTrain: {X_tr.shape}, 라벨분포 {np.bincount(y_tr)}")
print(f"Test:  {X_te.shape}, 라벨분포 {np.bincount(y_te)}")  # 둘 다 0 아님 확인!

X_tr = X_tr[..., np.newaxis]
X_te = X_te[..., np.newaxis]

# ---------- 4. 1D-CNN 모델 ----------
def build_model(n_feat):
    m = models.Sequential([
        layers.Input(shape=(n_feat, 1)),
        layers.Conv1D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv1D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
    return m

model = build_model(len(feat_cols))
model.summary()

# 클래스 불균형 보정 (결함이 정상의 4배)
n0, n1 = np.bincount(y_tr)
cw = {0: (n0+n1)/(2*n0), 1: (n0+n1)/(2*n1)}
print("class_weight:", cw)

# ---------- 5. 학습 ----------
# 검증셋을 그룹 기반으로 분리 (train 베어링 중 일부를 검증용으로)
val_b = {'K005', 'KA09', 'KI07'}   # train 안에서 정상+결함 섞어 검증용 지정
val_mask_tr = df[tr_idx]['bearing_code'].isin(val_b).values

Xtr2, ytr2 = X_tr[~val_mask_tr], y_tr[~val_mask_tr]
Xval, yval = X_tr[val_mask_tr],  y_tr[val_mask_tr]
print(f"실학습 {Xtr2.shape}, 검증 {Xval.shape} 검증라벨 {np.bincount(yval)}")

model = build_model(len(feat_cols))
model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
              loss='binary_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True)

hist = model.fit(Xtr2, ytr2,
                 validation_data=(Xval, yval),
                 epochs=40, batch_size=256,
                 class_weight=cw, callbacks=[es], verbose=1)


# ---------- 6. 평가 (시뮬레이터 KPI 기준) ----------
prob = model.predict(X_te).ravel()
pred = (prob >= 0.5).astype(int)

acc = accuracy_score(y_te, pred)
mf1 = f1_score(y_te, pred, average='macro')
cm  = confusion_matrix(y_te, pred)   # [[TN,FP],[FN,TP]]
tn, fp, fn, tp = cm.ravel()

fnr = fn / (fn + tp) if (fn+tp) else 0   # 미탐(놓친 결함) - 가장 위험
far = fp / (fp + tn) if (fp+tn) else 0   # 오경보(정상을 결함으로)

print("\n=== 성능 (시뮬레이터 목표 대비) ===")
print(f"Accuracy : {acc*100:.2f}%   (목표 98%)")
print(f"Macro F1 : {mf1*100:.2f}%   (목표 97%)")
print(f"FNR(미탐): {fnr*100:.2f}%   (목표 <2%)")
print(f"FAR(오경보): {far*100:.2f}%  (목표 <1%)")
print("\n혼동행렬 [[TN,FP],[FN,TP]]:\n", cm)
print("\n", classification_report(y_te, pred,
        target_names=['Healthy','Fault']))

df = pd.read_csv(PATH, low_memory=False)

# 6번 평가 다음에 추가 — 임계값별 FNR/FAR 곡선
print("\n=== 임계값 튜닝 ===")
print(f"{'thr':>5} {'FNR%':>7} {'FAR%':>7} {'Acc%':>7} {'MacroF1%':>9}")
for thr in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    p = (prob >= thr).astype(int)
    c = confusion_matrix(y_te, p)
    tn2, fp2, fn2, tp2 = c.ravel()
    fnr2 = fn2/(fn2+tp2)*100
    far2 = fp2/(fp2+tn2)*100
    acc2 = (tn2+tp2)/c.sum()*100
    mf12 = f1_score(y_te, p, average='macro')*100
    print(f"{thr:>5} {fnr2:>7.2f} {far2:>7.2f} {acc2:>7.2f} {mf12:>9.2f}")




Mounted at /content/drive
파일 존재 여부: True
전체 컬럼: ['vib_rms', 'vib_mean', 'vib_std', 'vib_kurtosis', 'vib_skewness', 'vib_peak', 'vib_peak_to_peak', 'vib_crest_factor', 'vib_shape_factor', 'vib_impulse_factor', 'vib_spectral_centroid', 'vib_spectral_bandwidth', 'vib_spectral_energy', 'vib_dominant_freq', 'vib_band_energy_0', 'vib_band_energy_1', 'vib_band_energy_2', 'vib_band_energy_3', 'vib_band_energy_4', 'cur1_rms', 'cur1_mean', 'cur1_std', 'cur1_kurtosis', 'cur1_skewness', 'cur1_peak', 'cur1_peak_to_peak', 'cur1_crest_factor', 'cur1_shape_factor', 'cur1_impulse_factor', 'cur1_spectral_centroid', 'cur1_spectral_bandwidth', 'cur1_spectral_energy', 'cur1_dominant_freq', 'cur1_band_energy_0', 'cur1_band_energy_1', 'cur1_band_energy_2', 'cur1_band_energy_3', 'cur1_band_energy_4', 'cur2_rms', 'cur2_mean', 'cur2_std', 'cur2_kurtosis', 'cur2_skewness', 'cur2_peak', 'cur2_peak_to_peak', 'cur2_crest_factor', 'cur2_shape_factor', 'cur2_impulse_factor', 'cur2_spectral_centroid', 'cur2_spectral_b

/tmp/ipykernel_809/2635552029.py:52: DtypeWarning: Columns (64,66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/paderborn_features_scaled_no_overlap.csv')


=== COLUMNS ===
0: vib_rms
1: vib_mean
2: vib_std
3: vib_kurtosis
4: vib_skewness
5: vib_peak
6: vib_peak_to_peak
7: vib_crest_factor
8: vib_shape_factor
9: vib_impulse_factor
10: vib_spectral_centroid
11: vib_spectral_bandwidth
12: vib_spectral_energy
13: vib_dominant_freq
14: vib_band_energy_0
15: vib_band_energy_1
16: vib_band_energy_2
17: vib_band_energy_3
18: vib_band_energy_4
19: cur1_rms
20: cur1_mean
21: cur1_std
22: cur1_kurtosis
23: cur1_skewness
24: cur1_peak
25: cur1_peak_to_peak
26: cur1_crest_factor
27: cur1_shape_factor
28: cur1_impulse_factor
29: cur1_spectral_centroid
30: cur1_spectral_bandwidth
31: cur1_spectral_energy
32: cur1_dominant_freq
33: cur1_band_energy_0
34: cur1_band_energy_1
35: cur1_band_energy_2
36: cur1_band_energy_3
37: cur1_band_energy_4
38: cur2_rms
39: cur2_mean
40: cur2_std
41: cur2_kurtosis
42: cur2_skewness
43: cur2_peak
44: cur2_peak_to_peak
45: cur2_crest_factor
46: cur2_shape_factor
47: cur2_impulse_factor
48: cur2_spectral_centroid
49: cur2_s

/tmp/ipykernel_809/2635552029.py:84: DtypeWarning: Columns (64,66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH)


SHAPE: (99294, 71)
사용 특징 수: 57
라벨 분포(0=정상,1=결함): [19209 80085]
정상 베어링 6종: ['K001', 'K002', 'K003', 'K004', 'K005', 'K006']
결함 베어링 25종: 25개

Train 정상: ['K003', 'K006', 'K002', 'K001', 'K005']
Test  정상: ['K004']

Train: (80064, 57), 라벨분포 [16009 64055]
Test:  (19230, 57), 라벨분포 [ 3200 16030]


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 57, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 57, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 57, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 57, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,945 (42.75 KB)

 Trainable params: 10,753 (42.00 KB)

 Non-trainable params: 192 (768.00 B)

class_weight: {0: np.float64(2.5005934162033854), 1: np.float64(0.6249629224884865)}
실학습 (70457, 57, 1), 검증 (9607, 57, 1) 검증라벨 [3200 6407]
Epoch 1/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 16s 47ms/step - accuracy: 0.8144 - loss: 0.3303 - val_accuracy: 0.3767 - val_loss: 1.2960
Epoch 2/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.9603 - loss: 0.1070 - val_accuracy: 0.5546 - val_loss: 2.9900
Epoch 3/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - accuracy: 0.9826 - loss: 0.0530 - val_accuracy: 0.6406 - val_loss: 3.6537
Epoch 4/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.9891 - loss: 0.0345 - val_accuracy: 0.6055 - val_loss: 3.9724
Epoch 5/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 13s 48ms/step - accuracy: 0.9923 - loss: 0.0261 - val_accuracy: 0.6402 - val_loss: 4.2152
Epoch 6/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 13s 47ms/step - accuracy: 0.9919 - loss: 0.0247 - val_accuracy: 0.6456 - val_loss: 4.1018
Epoch 7/40
276/276 ━━━━━━━━━━━━━━━━━━━━ 13s 49ms/step - accuracy: 0.9928 - loss: 